In [1]:
import numpy as np
from scipy.io import wavfile

# Load the WAV file
sample_rate, audio_data = wavfile.read('sample_coaching_session.wav')

# Now audio_data is a numpy array
print(f"Sample rate: {sample_rate} Hz")
print(f"Audio shape: {audio_data.shape}")
print(f"Data type: {audio_data.dtype}")

# If you need to normalize the audio to range [-1.0, 1.0]
# (especially useful if you'll be doing signal processing)
if audio_data.dtype == np.int16:
    audio_normalized = audio_data.astype(np.float32) / 32768.0
elif audio_data.dtype == np.int32:
    audio_normalized = audio_data.astype(np.float32) / 2147483648.0
    
# If you only need one channel from stereo audio
# (Your file is stereo based on your earlier output)
if len(audio_data.shape) > 1 and audio_data.shape[1] == 2:
    # Extract left channel only
    left_channel = audio_data[:, 0]
    
    # Or average both channels to get mono
    mono_audio = audio_data.mean(axis=1)

Sample rate: 48000 Hz
Audio shape: (99962880, 2)
Data type: int16


In [2]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

CUDA_DEVICE=1
device = f"cuda:{CUDA_DEVICE}"
TORCH_DTYPE = torch.float16

TRANSCRIPT_MODEL_ID = "models/whisper-large-v3-turbo"
transcript_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    TRANSCRIPT_MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
transcript_model.to(device)
transcript_processor = AutoProcessor.from_pretrained(TRANSCRIPT_MODEL_ID)

transcript_pipe = pipeline(
    "automatic-speech-recognition",
    model=transcript_model,
    tokenizer=transcript_processor.tokenizer,
    feature_extractor=transcript_processor.feature_extractor,
    torch_dtype=TORCH_DTYPE,
    device=device,
)

def custom_transcribe(audio_data, sample_rate):
    """Transcribes the voice with timestamps"""
    audio_input = {"raw": audio_data, "sampling_rate": sample_rate}
    output = transcript_pipe(audio_input, return_timestamps="word")
    return [ chunk["text"] for chunk in output["chunks"]]

/home/pavlo/miniconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


RuntimeError: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW